# Activity 4 Concepts: Resampling and Metrics

Activity 4 asks you to write a Streamlit app that loads the NYC taxi data, resamples it to daily
totals, and displays the busiest and quietest day as metric tiles. If you have never called
`.resample()` or built an f-string number format before, doing that for the first time inside a
`.py` file with no cell output is a rough way to learn it.

This notebook walks through every piece of that data logic on its own, with visible output at each
step. By the end, you will have run every line the app needs, and you will understand why each one
is there. Then you carry the code into `activity_4_first_app.py` almost unchanged.

Import pandas. Expect no output, the cell just runs.

In [ ]:
import pandas as pd

Load `data/nyc_taxi.csv` with `parse_dates` so the timestamp column becomes real dates, and
`index_col` so pandas uses it as the DataFrame's index instead of a plain column. Expect a
DataFrame with one column, `value`, indexed by timestamp.

In [ ]:
raw = pd.read_csv("data/nyc_taxi.csv", parse_dates=["timestamp"], index_col="timestamp")
raw.head()

Look at the spacing between rows before you do anything else with this data. Expect the gap
between the first two timestamps to be 30 minutes, and the same to be true throughout the file.
The raw data is HALF-HOURLY. That single fact is the reason everything else in this notebook
exists: a dashboard reporting "busiest day" needs one number per day, not 48 numbers per day.

In [ ]:
print(raw.index[0], "to", raw.index[1])
print(raw.index[1] - raw.index[0])
print(raw.shape)

`.resample("D")` groups the half-hourly rows into daily buckets, but on its own it does not
produce numbers. Expect it to print a `DatetimeIndexResampler` object, not a table of values. A
resampler is a plan for grouping, waiting for you to say how to combine each group.

In [ ]:
raw["value"].resample("D")

Add `.sum()` to tell the resampler how to combine each day's half-hourly rows: add them up.
Expect the row count to drop from the raw count you saw above to 215, one row per calendar day
from 2014-07-01 to 2015-01-31.

In [ ]:
daily = raw["value"].resample("D").sum()
print(len(daily), "days")
print(daily.index.min().date(), "to", daily.index.max().date())
daily.head()

Why `.sum()` and not `.mean()`? Each half-hourly value is a COUNT of rides in that 30-minute
window. Summing 48 counts gives the total rides that day, the quantity the app reports. Averaging
them instead gives the typical half-hourly rate, a much smaller and different number. Compare the
two for the first three days to see how different they are.

In [ ]:
compare = pd.DataFrame({
    "sum (total rides)": raw["value"].resample("D").sum(),
    "mean (average half-hour rate)": raw["value"].resample("D").mean(),
})
compare.head(3)

`.max()` returns the biggest daily total in the whole series, as a plain number. Expect a
single value: the highest ride count any one day reached.

In [ ]:
daily.max()

`.min()` works the same way, on the low end. Expect a single value: the lowest ride count any
one day reached.

In [ ]:
daily.min()

`.max()` tells you the busiest day's ride count, but not which day it was. This is the
distinction that trips people up: `.max()` returns the VALUE. `.idxmax()` returns the INDEX
(here, the date) where that value occurred. The app needs both: the number for the metric tile,
the date for the caption underneath it. Expect a single Timestamp.

In [ ]:
daily.idxmax()

`.idxmin()` is the same idea for the low end: the date the quietest day happened, not the
count. Expect a single Timestamp.

In [ ]:
daily.idxmin()

Put the four together as a sentence, and you have both KPI tiles the app needs. Confirm the
value and the date line up: the count from `.max()` should be the value stored on the date from
`.idxmax()`, and the same for the minimum.

In [ ]:
print("Busiest day:", daily.max(), "rides, on", daily.idxmax().date())
print("Quietest day:", daily.min(), "rides, on", daily.idxmin().date())

Now the formatting problem. `st.metric` in the app displays a string, not a raw number, and a
raw number is not something you would show a stakeholder. Look at the average ride count as a
plain f-string with no formatting at all. Expect a long, hard-to-read decimal, this is what
`{value}` alone gives you.

In [ ]:
value = daily.mean()
print(value)
print(f"{value}")

Add `,` inside the format spec. Expect the same ugly decimal tail, but now with a thousands
separator in the whole-number part. The comma is exactly that: a thousands separator, nothing
else. It does not touch the decimals.

In [ ]:
print(f"{value:,}")

Add `.0f` after the comma. Expect a clean whole number with a thousands separator and no
decimal point at all. `.0f` means "fixed-point notation, 0 digits after the decimal", so this
rounds to the nearest whole number and drops the rest. `{value:,.0f}` is the full spec: comma
first for the separator, then `.0f` for zero decimal places.

In [ ]:
print(f"{value:,.0f}")

Apply that exact format spec, `:,.0f`, to the busiest and quietest day counts. This is the
same string the app builds for its two `st.metric` calls.

In [ ]:
print(f"Busiest day: {daily.max():,.0f} rides")
print(f"Quietest day: {daily.min():,.0f} rides")

These five lines are the whole data layer of the app you are about to write:

```python
raw = pd.read_csv(path, parse_dates=["timestamp"], index_col="timestamp")
daily = raw["value"].resample("D").sum()
st.write(f"{len(daily):,} days, {daily.index.min().date()} to {daily.index.max().date()}")
left.metric("Busiest day", f"{daily.max():,.0f}", help=str(daily.idxmax().date()))
right.metric("Quietest day", f"{daily.min():,.0f}", help=str(daily.idxmin().date()))
```

Everything else in `activity_4_first_app.py` is Streamlit layout (`st.title`, `st.columns`,
`st.line_chart`, `st.expander`) wrapped around the five lines above. You already ran every one of
them in this notebook and saw exactly what they produce.

**Try changing this and re-run, no new code needed:**

1. Change `.resample("D")` to `.resample("W")` in the `daily` cell and re-run just that cell and
   the cells after it. How many rows do you get now, and does `.max()` still return the busiest
   total once you are summing whole weeks instead of days?
2. In the formatting cell, change `:,.0f` to `:,.2f`. What shows up after the decimal point now,
   and why would that be a worse choice for a ride-count metric than a stock price metric?